In [ ]:
import pandas as pd
import pandas_ta as ta
import numpy as np
import xgboost as xgb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

symbols = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT', 'BNB/USDT']
import os
in_notebook_dir = os.path.basename(os.getcwd()) == 'notebooks'
cache_dir = '../data' if in_notebook_dir else 'data'



def prepare_data(df):
    df = df.copy()
    
    df['EMA9'] = ta.ema(df['close'], length=9)
    df['EMA21'] = ta.ema(df['close'], length=21)
    df['EMA_CROSS'] = (df['EMA9'] - df['EMA21']) / df['EMA21']
    
    adx = ta.adx(df['high'], df['low'], df['close'], length=14)
    if adx is not None:
        df['ADX'], df['DMP'], df['DMN'] = adx.iloc[:, 0], adx.iloc[:, 1], adx.iloc[:, 2]
    else:
        df['ADX'], df['DMP'], df['DMN'] = 0, 0, 0
        
    st = ta.supertrend(df['high'], df['low'], df['close'], length=10, multiplier=3.0)
    df['SUPERTREND_DIR'] = st.iloc[:, 1] if st is not None else 0

    df['RSI'] = ta.rsi(df['close'], length=14)
    df['ATR'] = ta.atr(df['high'], df['low'], df['close'], length=14)
    macd = ta.macd(df['close'])
    df['MACD'] = macd.iloc[:, 0] if macd is not None else 0
    df['MACD_HIST'] = macd.iloc[:, 1] if macd is not None else 0
    bb = ta.bbands(df['close'], length=20, std=2.0)
    df['BB_WIDTH'] = (bb.iloc[:, 2] - bb.iloc[:, 0]) / bb.iloc[:, 1] if bb is not None else 0
    df['BB_POS'] = (df['close'] - bb.iloc[:, 0]) / (bb.iloc[:, 2] - bb.iloc[:, 0]) if bb is not None else 0.5
    df['RET_1'] = df['close'].pct_change(1)
    df['RET_3'] = df['close'].pct_change(3)
    
    for col in ['RSI', 'ADX', 'MACD', 'BB_WIDTH']:
        df[col + '_Z'] = (df[col] - df[col].rolling(200).mean()) / df[col].rolling(200).std()
        
    df.fillna(0, inplace=True)
    
    target_long, target_short = [], []
    close, high, low, atr = df['close'].values, df['high'].values, df['low'].values, df['ATR'].values
    n = len(df)
    max_hold = 30
    
    for i in range(n):
        if i + max_hold >= n or np.isnan(atr[i]):
            target_long.append(0); target_short.append(0); continue
        c, cur_atr = close[i], atr[i]
        
        tp_price_l, sl_price_l = c + (cur_atr * 2.5), c - (cur_atr * 1.5)
        tp_price_s, sl_price_s = c - (cur_atr * 2.5), c + (cur_atr * 1.5)
        hit_l, hit_s = 0, 0
        
        for j in range(1, max_hold + 1):
            curr_h, curr_l = high[i+j], low[i+j]
            if hit_l == 0:
                if curr_l <= sl_price_l: hit_l = -1
                elif curr_h >= tp_price_l: hit_l = 1
            if hit_s == 0:
                if curr_h >= sl_price_s: hit_s = -1
                elif curr_l <= tp_price_s: hit_s = 1
            if hit_l != 0 and hit_s != 0: break
                
        target_long.append(1 if hit_l == 1 else 0)
        target_short.append(1 if hit_s == 1 else 0)
        
    df['TARGET_L'] = target_long
    df['TARGET_S'] = target_short
    df.dropna(inplace=True)
    return df

prepared_dfs = {}
for sym in symbols:
    cache_file = f"{cache_dir}/{sym.replace('/', '_')}_15m_10000.csv"
    if os.path.exists(cache_file):
        df_raw = pd.read_csv(cache_file, index_col='timestamp', parse_dates=True)
        prepared_dfs[sym] = prepare_data(df_raw.tail(10000))
        print(f"Cargado {sym} exitosamente")
    else:
        print(f"No se encontró {sym}")


In [ ]:
import json
import os

in_notebook_dir = os.path.basename(os.getcwd()) == 'notebooks'
config_path = '../config/current_params.json' if in_notebook_dir else 'config/current_params.json'
if os.path.exists(config_path):


    with open(config_path, 'r') as f:
        params_per_symbol = json.load(f)
else:
    params_per_symbol = {}
    print("ADVERTENCIA: No se encontro config/current_params.json")

features = ['EMA_CROSS', 'DMP', 'DMN', 'SUPERTREND_DIR', 'MACD_HIST', 'BB_POS', 'RET_1', 'RET_3', 'RSI_Z', 'ADX_Z', 'MACD_Z', 'BB_WIDTH_Z']

# Sincronizar DataFrames para backtest cronológico
common_index = None
for sym in symbols:
    if sym in prepared_dfs:
        if common_index is None:
            common_index = prepared_dfs[sym].index
        else:
            common_index = common_index.intersection(prepared_dfs[sym].index)

for sym in symbols:
    if sym in prepared_dfs:
        prepared_dfs[sym] = prepared_dfs[sym].loc[common_index]

val_start = int(len(common_index) * 0.80)
val_end = len(common_index)

models = {}
for sym in symbols:
    if sym not in prepared_dfs: continue
    print(f"Entrenando modelos para {sym}...")
    df = prepared_dfs[sym]
    train_df = df.iloc[:val_start]
    
    p = params_per_symbol.get(sym, {})
    if not p:
        print(f"Sin parametros para {sym}, omitiendo.")
        continue
        
    X_train = train_df[features]
    yl_train = train_df['TARGET_L']
    ys_train = train_df['TARGET_S']
    
    ml = xgb.XGBClassifier(n_estimators=100, max_depth=p.get('max_depth', 3), learning_rate=p.get('learning_rate', 0.1), 
                           reg_alpha=p.get('reg_alpha', 1.0), reg_lambda=p.get('reg_lambda', 1.0), n_jobs=-1, random_state=42)
    ms = xgb.XGBClassifier(n_estimators=100, max_depth=p.get('max_depth', 3), learning_rate=p.get('learning_rate', 0.1), 
                           reg_alpha=p.get('reg_alpha', 1.0), reg_lambda=p.get('reg_lambda', 1.0), n_jobs=-1, random_state=42)
    ml.fit(X_train, yl_train)
    ms.fit(X_train, ys_train)
    models[sym] = {'ml': ml, 'ms': ms, 'params': p}

print("Modelos entrenados listos.")


In [ ]:
print("\nIniciando Backtest Global Sincronizado con Asignacion Dinamica...")
initial_capital = 250.0
available_capital = initial_capital
portfolio_value = []
open_trades = []

COM = 0.0004
slippage = 0.0003
max_hold = 30
max_risk_per_trade = 0.30 # NUNCA arriesgar mas del 30% del capital total en un solo trade

trade_history = []

for i in range(val_start, val_end - max_hold):
    current_time = common_index[i]
    
    # 1. Actualizar/Cerrar operaciones abiertas
    trades_to_keep = []
    for trade in open_trades:
        sym = trade['symbol']
        df = prepared_dfs[sym]
        curr_h, curr_l, curr_c = df['high'].iloc[i], df['low'].iloc[i], df['close'].iloc[i]
        
        # Check Trailing Stop Activation
        if trade['is_long']:
            if not trade['ts_activated'] and curr_h >= trade['ts_trigger']:
                trade['ts_activated'] = True
            
            # Close conditions
            salida = None
            if curr_l <= trade['sl_price']: salida = trade['sl_price'] * (1 - slippage)
            elif curr_h >= trade['tp_price']: salida = trade['tp_price']
            
            # Update TS
            if trade['ts_activated'] and salida is None:
                nuevo_sl = curr_c - (df['ATR'].iloc[i] * trade['sl_mult'] * 0.7)
                if nuevo_sl > trade['sl_price']: trade['sl_price'] = nuevo_sl
        else:
            if not trade['ts_activated'] and curr_l <= trade['ts_trigger']:
                trade['ts_activated'] = True
                
            # Close conditions
            salida = None
            if curr_h >= trade['sl_price']: salida = trade['sl_price'] * (1 + slippage)
            elif curr_l <= trade['tp_price']: salida = trade['tp_price']
            
            # Update TS
            if trade['ts_activated'] and salida is None:
                nuevo_sl = curr_c + (df['ATR'].iloc[i] * trade['sl_mult'] * 0.7)
                if nuevo_sl < trade['sl_price']: trade['sl_price'] = nuevo_sl
                
        # Timeout condition
        if salida is None and i >= trade['entry_idx'] + max_hold:
            salida = curr_c * (1 - slippage if trade['is_long'] else 1 + slippage)
            
        if salida is not None:
            sign = 1.0 if trade['is_long'] else -1.0
            pnl_pct = (salida - trade['entrada']) / trade['entrada'] * sign - COM * 2
            ganancia_usd = trade['pos_size'] * pnl_pct
            
            available_capital += trade['pos_size'] + ganancia_usd
            trade['pnl'] = ganancia_usd
            trade['exit_time'] = current_time
            trade['exit_price'] = salida
            trade['pnl_pct'] = pnl_pct
            trade_history.append(trade)
        else:
            trades_to_keep.append(trade)
            
    open_trades = trades_to_keep
    
    # Calcular valor total del portafolio (capital libre + valor actual de posiciones abiertas)
    current_portfolio_value = available_capital
    for trade in open_trades:
        sym = trade['symbol']
        curr_c = prepared_dfs[sym]['close'].iloc[i]
        sign = 1.0 if trade['is_long'] else -1.0
        pnl_pct = (curr_c - trade['entrada']) / trade['entrada'] * sign - COM * 2
        current_portfolio_value += trade['pos_size'] * (1 + pnl_pct)
        
    portfolio_value.append({'timestamp': current_time, 'capital': current_portfolio_value})
    
    # 2. Buscar nuevas entradas (solo si hay al menos $20 de capital disponible)
    if available_capital < 20:
        continue
        
    for sym, model_data in models.items():
        # Prevent opening multiple trades on the same coin at the same time
        if any(t['symbol'] == sym for t in open_trades):
            continue
            
        df = prepared_dfs[sym]
        X = df[features].iloc[[i]]
        prob_long = model_data['ml'].predict_proba(X)[0][1]
        prob_short = model_data['ms'].predict_proba(X)[0][1]
        
        p = model_data['params']
        c, cur_atr = df['close'].iloc[i], df['ATR'].iloc[i]
        
        is_l, is_s = prob_long > p['confidence'], prob_short > p['confidence']
        
        if is_l or is_s:
            es_long = is_l
            prob_final = prob_long if es_long else prob_short
            
            # --- ASIGNACION DINAMICA DE CAPITAL (KELLY / ML CONFIDENCE) ---
            entrada = c * (1 + slippage) if es_long else c * (1 - slippage)
            sl_dist = cur_atr * p['sl_mult']
            tp_dist = cur_atr * p['tp_mult']
            sl_price = entrada - sl_dist if es_long else entrada + sl_dist
            tp_price = entrada + tp_dist if es_long else entrada - tp_dist
            
            riesgo_real_pct = abs(entrada - sl_price) / entrada
            
            # Tamano de posicion dinamico
            ideal_pos_size = (current_portfolio_value * p['risk_pct'] * prob_final) / max(riesgo_real_pct, 0.001)
            
            pos_size = min(ideal_pos_size, current_portfolio_value * max_risk_per_trade)
            pos_size = min(pos_size, available_capital)
            
            if pos_size < 10: 
                continue
                
            available_capital -= pos_size
            
            activation_dist = tp_dist * 0.4
            ts_trigger = entrada + activation_dist if es_long else entrada - activation_dist
            
            open_trades.append({
                'symbol': sym,
                'is_long': es_long,
                'type': 'LONG' if es_long else 'SHORT',
                'entrada': entrada,
                'entry_price': entrada,
                'entry_time': current_time,
                'sl_price': sl_price,
                'tp_price': tp_price,
                'ts_trigger': ts_trigger,
                'ts_activated': False,
                'sl_mult': p['sl_mult'],
                'pos_size': pos_size,
                'entry_idx': i,
                'prob': prob_final
            })

portfolio_df = pd.DataFrame(portfolio_value).set_index('timestamp')
trades_df = pd.DataFrame(trade_history)

if not trades_df.empty:
    print(f'=== ALGORITMO MULTI-MODELO OPTIMIZADO (CAPITAL DINAMICO) ===')
    print(f'Trades realizados en conjunto: {len(trades_df)}')
    win_rate = (trades_df["pnl"] > 0).mean() * 100
    print(f'Win Rate Global: {win_rate:.2f}%')
    roi_total = ((current_portfolio_value - initial_capital) / initial_capital) * 100
    print(f'Capital Inicial: $250.00')
    print(f'Capital Final: ${current_portfolio_value:.2f}')
    print(f'ROI Total: {roi_total:.2f}%')
    
    print("\nDesglose de PnL por Moneda:")
    for sym in symbols:
        sym_trades = trades_df[trades_df['symbol'] == sym]
        if sym_trades.empty: continue
        wins = (sym_trades['pnl'] > 0).sum()
        total_pnl = sym_trades['pnl'].sum()
        wr = wins / len(sym_trades) * 100
        avg_pos = sym_trades['pos_size'].mean()
        print(f"{sym}: {len(sym_trades)} trades, WinRate: {wr:.1f}%, PnL: ${total_pnl:.2f}, Inversion Promedio: ${avg_pos:.2f}")
else:
    print("No se realizaron trades.")

# Grafico de Equidad Global
plt.figure(figsize=(14, 7))
plt.plot(portfolio_df.index, portfolio_df['capital'], label='Capital Unificado ($)', color='blue')
plt.axhline(y=initial_capital, color='red', linestyle='--', alpha=0.5, label='Capital Inicial')
plt.title('Curva de Rendimiento Sincronizada y Asignacion Dinamica')
plt.xlabel('Fecha')
plt.ylabel('USDT')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
for symbol in symbols:
    if symbol not in prepared_dfs: continue
    df_sym = prepared_dfs[symbol]
    
    df_plot = df_sym.tail(1920)
    
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03, row_heights=[0.7, 0.3])

    fig.add_trace(go.Candlestick(x=df_plot.index,
                    open=df_plot['open'],
                    high=df_plot['high'],
                    low=df_plot['low'],
                    close=df_plot['close'],
                    name='Precio'), row=1, col=1)

    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['EMA9'], line=dict(color='orange', width=1), name='EMA9'), row=1, col=1)
    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['EMA21'], line=dict(color='purple', width=2), name='EMA21'), row=1, col=1)

    if not trades_df.empty:
        sym_trades = trades_df[trades_df['symbol'] == symbol]
        for _, trade in sym_trades.iterrows():
            if trade['entry_time'] in df_plot.index:
                color = 'green' if trade['pnl'] > 0 else 'red'
                marker_symbol = 'triangle-up' if trade['type'] == 'LONG' else 'triangle-down'
                
                fig.add_trace(go.Scatter(x=[trade['entry_time']], y=[trade['entry_price']],
                                         mode='markers', marker=dict(symbol=marker_symbol, size=15, color=color),
                                         name=f"Entrada {trade['type']} (${trade['pnl']:.2f})"), row=1, col=1)
                                         
                if trade['exit_time'] in df_plot.index:
                    fig.add_trace(go.Scatter(x=[trade['exit_time']], y=[trade['exit_price']],
                                             mode='markers', marker=dict(symbol='x', size=10, color='yellow'),
                                             showlegend=False), row=1, col=1)

    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['RSI'], name='RSI', line=dict(color='blue')), row=2, col=1)
    fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=1)
    fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1)

    fig.update_layout(title=f'Backtest {symbol} (Asignacion Dinamica)', xaxis_rangeslider_visible=False, template='plotly_dark')
    fig.show()
